# Transaction Anomaly Dashboard - Segmentasi & Deteksi Anomali Multivariat

## Tujuan Proyek
Membangun pipeline *unsupervised machine learning* untuk **Transaction Anomaly Dashboard**:
1. **Segmentasi Perilaku (K-Means, `k = 2`)** pada 5 fitur perilaku (`TransactionAmount`, `CustomerAge`, `TransactionDuration`, `AccountBalance`, `Balance_Ratio`).
2. **Deteksi Anomali Multivariat (Ensemble 3 Model)** memakai `IsolationForest`, `LocalOutlierFactor`, dan `PCA Reconstruction Error` pada 8 fitur (termasuk fitur temporal `Days_Since_Prev_Tx` dan kecepatan transaksi `Amount_Per_Second`).
3. **Risiko Login (`LoginAttempts >= 3`)** sebagai *guardrail* tambahan.

**Sumber Dataset:** Dataset latihan Dicoding (adaptasi Kaggle *Bank Transaction Dataset for Fraud Detection*) - data sintetis transaksi bank kota AS dalam satuan USD, **tanpa label *fraud* asli (*ground truth*)**.

**Cara pakai:** jalankan notebook dari atas ke bawah (*Run All*). Sel terakhir menulis `data_clustering_clean.csv`, `scaler_clustering.joblib`, `kmeans_model.joblib`, `anomaly_bundle.joblib`, dan `k_evaluation.csv` yang dibaca langsung oleh `app.py`, sehingga semua angka di dashboard dan README berasal dari notebook ini.

---

# 1. Import Library & Cek Data Awal
Siapkan library yang dipakai dan lihat kondisi data mentah sebelum dibersihkan.

In [ ]:
# Hapus tanda # jika library belum terinstal
# %pip install pandas matplotlib seaborn scikit-learn joblib openpyxl

import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import joblib

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

print("Library siap.")

## 1.1 Muat Dataset
Baca data sintetis transaksi bank dari `data/raw.xlsx` (atau `raw.xlsx` / URL cadangan Google Sheets jika file lokal tidak tersedia), lalu salin ke `df` agar data mentah tetap aman.

In [ ]:
local_candidates = ['data/raw.xlsx', 'raw.xlsx']
local_path = next((p for p in local_candidates if os.path.exists(p)), None)
backup_url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vTbg5WVW6W3c8SPNUGc3A3AL-AG32TPEQGpdzARfNICMsLFI0LQj0jporhsLCeVhkN5AoRsTkn08AYl/pub?output=csv'

if local_path:
    df_raw = pd.read_excel(local_path)
    print(f"Data dimuat dari: {local_path}")
else:
    df_raw = pd.read_csv(backup_url)
    print("Data dimuat dari URL Google Sheets.")

df = df_raw.copy()
print(f"Ukuran data: {df.shape[0]} baris, {df.shape[1]} kolom")
df.head()

## 1.2 Cek Tipe Data, Missing Values, dan Duplikat
Periksa tipe tiap kolom, jumlah data kosong, dan baris ganda.

In [ ]:
print(f"Baris duplikat: {df.duplicated().sum()}\n")

data_summary = pd.DataFrame({
    'Tipe Data': df.dtypes,
    'Non-Null': df.notnull().sum(),
    'Missing': df.isnull().sum(),
    'Missing (%)': (df.isnull().sum() / len(df)) * 100,
    'Nilai Unik': df.nunique()
})
display(data_summary)

## 1.3 Statistik Deskriptif Awal
Cek rentang angka numerik (waspadai nilai tidak wajar) dan kategori paling sering muncul pada kolom teks.

In [ ]:
print("=== Fitur Numerik ===")
display(df.describe().T)

print("\n=== Fitur Kategorikal ===")
display(df.describe(include=['object', 'string']).T)

---

# 2. Pembersihan Data (Data Cleaning)
1. **Hitung `Days_Since_Prev_Tx` lebih dulu**: selisih `TransactionDate` dan `PreviousTransactionDate` (hari) dihitung **sebelum** duplikat dibuang, supaya tiap baris tetap sejajar dengan tanggalnya sendiri.
2. **Hapus duplikat**: buang `21` baris ganda agar tidak membobot berlebih.
3. **Hapus kolom tidak relevan**: buang kolom ID & IP (`TransactionID`, `AccountID`, `DeviceID`, `MerchantID`, `IP Address`) dan kolom tanggal mentah (`TransactionDate`, `PreviousTransactionDate`).
4. **Isi nilai kosong (imputasi)**: pakai **median** untuk kolom numerik (tahan *outlier*) dan **modus** untuk kolom teks, karena porsi data kosong kecil (`< 1.2%` per kolom).

In [ ]:
initial_rows = len(df)

# Jeda hari dihitung SEBELUM drop_duplicates + reset_index, agar baris tidak bergeser
# terhadap data mentah. NaN diimputasi median pada langkah 2.1.
df['Days_Since_Prev_Tx'] = (
    (pd.to_datetime(df['TransactionDate']) - pd.to_datetime(df['PreviousTransactionDate']))
    .dt.total_seconds() / 86400.0
)

df = df.drop_duplicates().reset_index(drop=True)
print(f"Duplikat dihapus: {initial_rows - len(df)} baris (sisa {len(df)} baris)")

cols_to_drop = [
    'TransactionID', 'AccountID', 'DeviceID', 'MerchantID',
    'IP Address', 'PreviousTransactionDate', 'TransactionDate'
]
df = df.drop(columns=cols_to_drop)

print(f"Kolom tersisa ({df.shape[1]}): {df.columns.tolist()}")

## 2.1 Imputasi Nilai Kosong
Isi nilai kosong pada kolom numerik dengan median dan kolom kategorikal dengan modus.

In [ ]:
numerical_cols = df.select_dtypes(include='number').columns.tolist()
categorical_cols = df.select_dtypes(include=['object', 'string']).columns.tolist()

for col in numerical_cols:
    df[col] = df[col].fillna(df[col].median())

for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print("Missing values setelah imputasi:")
print(df.isnull().sum())
print(f"\nUkuran data bersih: {df.shape[0]} baris, {df.shape[1]} kolom")
df.head()

---

# 3. Feature Engineering, Aturan Risiko Login, & EDA
1. **`Balance_Ratio`**: rasio `TransactionAmount / AccountBalance` untuk melihat seberapa besar porsi saldo yang ditarik tiap transaksi.
2. **`High_Login_Risk`**: aturan eksplisit `LoginAttempts >= 3` sebagai indikator risiko login terpisah dari clustering (bukan validasi *fraud* karena dataset tidak memiliki label *fraud* asli).
3. **EDA**: cek distribusi data, *outlier* alami, dan korelasi antar fitur numerik.

In [ ]:
df['Balance_Ratio'] = df['TransactionAmount'] / df['AccountBalance']
df['High_Login_Risk'] = df['LoginAttempts'] >= 3
numerical_cols = df.select_dtypes(include='number').columns.tolist()

print(f"Fitur numerik ({len(numerical_cols)}): {numerical_cols}")
print(f"Transaksi berisiko login (LoginAttempts >= 3): {int(df['High_Login_Risk'].sum())} dari {len(df)} ({df['High_Login_Risk'].mean() * 100:.2f}%)\n")
print("=== Statistik Fitur Numerik ===")
display(df[numerical_cols].describe().T)

print("\n=== Sebaran LoginAttempts ===")
login_counts = pd.DataFrame({
    'Jumlah': df['LoginAttempts'].value_counts().sort_index(),
    'Persentase (%)': df['LoginAttempts'].value_counts(normalize=True).sort_index() * 100
})
display(login_counts)

## 3.1 Distribusi Fitur Numerik
Histogram melihat bentuk sebaran (normal atau miring); boxplot menandai *outlier* alami yang sengaja dipertahankan.

In [ ]:
def plot_numeric_grid(data, cols, kind, title, figsize):
    """Plot grid histogram atau boxplot (3 kolom) untuk daftar fitur numerik."""
    n_rows = int(np.ceil(len(cols) / 3))
    fig, axes = plt.subplots(n_rows, 3, figsize=(figsize[0], figsize[1] * n_rows / 2))
    axes = np.atleast_1d(axes).flatten()
    for ax, col in zip(axes, cols):
        if kind == 'hist':
            sns.histplot(data[col], bins=25, kde=True, color='steelblue', ax=ax)
            ax.set_ylabel('Frekuensi')
        else:
            sns.boxplot(x=data[col], color='lightcoral', ax=ax)
        ax.set_title(col, fontsize=12, fontweight='bold')
        ax.set_xlabel(col)
    for ax in axes[len(cols):]:
        ax.axis('off')
    plt.suptitle(title, fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

plot_numeric_grid(df, numerical_cols, 'hist', 'Histogram Distribusi Fitur Numerik', (18, 9))
plot_numeric_grid(df, numerical_cols, 'box', 'Boxplot Sebaran & Outlier Fitur Numerik', (18, 8))

## 3.2 Fitur Kategorikal & Korelasi
Cek proporsi kategori utama (`Location` tidak diplot karena ada `43` kota) dan korelasi antar fitur numerik agar tidak ada fitur ganda.

In [ ]:
main_cat_cols = ['TransactionType', 'Channel', 'CustomerOccupation']
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, main_cat_cols):
    order_vals = df[col].value_counts().index
    sns.countplot(data=df, x=col, order=order_vals, palette='viridis', ax=ax)
    ax.set_title(col, fontsize=12, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Jumlah Transaksi')

plt.suptitle('Distribusi Fitur Kategorikal Utama', fontsize=15, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 6))
sns.heatmap(df[numerical_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1, linewidths=0.5)
plt.title('Matriks Korelasi Fitur Numerik', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

# 4. Seleksi Fitur & Standarisasi (Preprocessing)
Tiga aturan preprocessing sebelum masuk ke **K-Means**:
1. **Keluarkan `LoginAttempts` dari fitur clustering**: memisahkan variabel diskrit `LoginAttempts` agar cluster tidak sekadar meniru aturan `LoginAttempts >= 3`, melainkan murni menangkap pola perilaku transaksi.
2. **Pakai 5 fitur numerik perilaku saja**: `TransactionAmount`, `CustomerAge`, `TransactionDuration`, `AccountBalance`, dan `Balance_Ratio`. Fitur teks tetap disimpan untuk analisis profil dan filter dashboard.
3. **Standarisasi (`StandardScaler`)**: samakan skala ke `mean = 0` dan `std = 1` supaya `AccountBalance` (ribuan USD) tidak mendominasi fitur lain.

In [ ]:
clustering_features = [
    'TransactionAmount', 'CustomerAge', 'TransactionDuration',
    'AccountBalance', 'Balance_Ratio'
]

scaler = StandardScaler()
df_scaled = pd.DataFrame(
    scaler.fit_transform(df[clustering_features]),
    columns=clustering_features
)

print(f"Ukuran df_scaled: {df_scaled.shape[0]} baris, {df_scaled.shape[1]} fitur")
df_scaled.head()

## 4.1 Cek Kesetaraan Varians
Pastikan semua fitur di `df_scaled` punya `mean = 0.00`, `std = 1.00`, dan `variance = 1.00`.

In [ ]:
scaled_summary = df_scaled.describe().T
scaled_summary['variance'] = df_scaled.var()

print("=== Statistik & Varians Setelah StandardScaler ===")
display(scaled_summary[['count', 'mean', 'std', 'variance', 'min', '50%', 'max']])

---

# 5. Penentuan Jumlah Cluster & Pemodelan K-Means
Uji `k = 2` sampai `k = 8` memakai dua metrik:
- **Elbow Method (*Inertia* / WCSS)**: pantau penurunan jarak dalam cluster.
- **Silhouette Score**: cari nilai tertinggi (rentang `-1` sampai `1`), sambil mencatat secara jujur bahwa nilai di bawah `0.5` menunjukkan struktur cluster antarfitur numerik masih lemah dan tumpang tindih.

In [ ]:
k_range = range(2, 9)
inertia_values = []
silhouette_values = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(df_scaled)
    inertia_values.append(km.inertia_)
    silhouette_values.append(silhouette_score(df_scaled, labels))

eval_df = pd.DataFrame({
    'k': list(k_range),
    'Inertia': inertia_values,
    'Silhouette Score': silhouette_values
})
print("=== Evaluasi k = 2 s/d 8 ===")
display(eval_df.assign(**{'Silhouette Score': eval_df['Silhouette Score'].map('{:.4f}'.format)}))

best_k = eval_df.loc[eval_df['Silhouette Score'].idxmax(), 'k']
best_sil = eval_df['Silhouette Score'].max()
print(f"\nk optimal: {best_k} (Silhouette Score = {best_sil:.4f})")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
metrics = [
    (inertia_values, 'o', 'steelblue', 'Elbow Method (Inertia)', 'Inertia'),
    (silhouette_values, 's', 'seagreen', 'Silhouette Score per k', 'Silhouette Score')
]
for ax, (vals, marker, color, title, ylabel) in zip(axes, metrics):
    ax.plot(k_range, vals, marker=marker, linewidth=2, color=color)
    ax.axvline(x=best_k, color='crimson', linestyle='--', label=f'k = {best_k}')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Jumlah Cluster (k)')
    ax.set_ylabel(ylabel)
    ax.legend()

plt.tight_layout()
plt.show()

## 5.1 Model Final (`k = 2`) & Visualisasi PCA 2D
`k = 2` memberi **Silhouette Score tertinggi** (`0.2634`) dibanding `k = 3..8`, meski pemisahannya tergolong lemah karena data perilaku numerik saling tumpang tindih. Pakai **PCA** untuk mereduksi 5 fitur menjadi 2 komponen (`PC1`, `PC2`) agar sebaran cluster dan *centroid* bisa divisualisasikan.

In [ ]:
optimal_k = 2
kmeans_model = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
cluster_labels = kmeans_model.fit_predict(df_scaled)

cluster_dist = pd.DataFrame({
    'Jumlah Transaksi': pd.Series(cluster_labels).value_counts().sort_index(),
    'Persentase (%)': pd.Series(cluster_labels).value_counts(normalize=True).sort_index() * 100
})
cluster_dist.index.name = 'Cluster'
print("=== Distribusi Anggota per Cluster ===")
display(cluster_dist)

pca = PCA(n_components=2, random_state=42)
pca_coords = pca.fit_transform(df_scaled)
centers_df = pd.DataFrame(kmeans_model.cluster_centers_, columns=clustering_features)
centroids_pca = pca.transform(centers_df)

df_pca = pd.DataFrame(pca_coords, columns=['PC1', 'PC2'])
df_pca['Cluster'] = [f'Cluster {c}' for c in cluster_labels]

plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=df_pca, x='PC1', y='PC2', hue='Cluster',
    palette=['#D97706', '#2563EB'],
    hue_order=['Cluster 0', 'Cluster 1'],
    alpha=0.75, s=55
)
plt.scatter(
    centroids_pca[:, 0], centroids_pca[:, 1],
    c='red', s=220, marker='X', edgecolors='black', linewidths=1.5, label='Centroid'
)

var_pc1, var_pc2 = pca.explained_variance_ratio_ * 100
plt.title('Sebaran Cluster 2D (PCA)', fontsize=14, fontweight='bold')
plt.xlabel(f'PC1 ({var_pc1:.1f}% Varians)')
plt.ylabel(f'PC2 ({var_pc2:.1f}% Varians)')
plt.legend(title='Kelompok')
plt.tight_layout()
plt.show()

---

# 6. Profiling Cluster, Interpretasi, & Ekspor Data
Kembalikan label `Cluster` ke data skala asli (`df`) agar profil tiap segmen mudah dibaca dalam satuan USD, tahun, dan detik.

In [ ]:
df['Cluster'] = cluster_labels

print("=== Rata-Rata (Mean) per Cluster ===")
display(df.groupby('Cluster')[clustering_features + ['LoginAttempts']].mean())

print("\n=== Nilai Tengah (Median) per Cluster ===")
display(df.groupby('Cluster')[clustering_features + ['LoginAttempts']].median())

print("\n=== Kategori Dominan (Modus) per Cluster ===")
display(df.groupby('Cluster')[categorical_cols].agg(lambda x: x.mode()[0]))

print("\n=== Cross-Tab Cluster vs High_Login_Risk (LoginAttempts >= 3) ===")
display(pd.crosstab(df['Cluster'], df['High_Login_Risk'], margins=True))

## 6.1 Perbandingan Fitur Utama Antar Cluster
Bandingkan 4 fitur perilaku (`CustomerAge`, `AccountBalance`, `Balance_Ratio`, `TransactionAmount`) antar segmen lewat boxplot.

In [ ]:
key_features = ['CustomerAge', 'AccountBalance', 'Balance_Ratio', 'TransactionAmount']
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

for ax, col in zip(axes.flatten(), key_features):
    sns.boxplot(
        data=df, x='Cluster', y=col,
        palette=['#D97706', '#2563EB'], ax=ax
    )
    ax.set_title(f'Perbandingan {col}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Cluster')
    ax.set_ylabel(col)

plt.suptitle('Profil Fitur Utama Antar Cluster', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 6.2 Interpretasi Segmen, Aturan Risiko Login, & Keterbatasan

### 0 - Nasabah Muda Aktif (`863` transaksi / `34.30%`)
* **Profil**: Rata-rata usia `27.30` tahun (dominan `Student`), saldo lebih rendah (`$1,584.53`), rata-rata transaksi lebih tinggi (`$347.07`), dan **`Balance_Ratio` `0.47`**. Terdapat `33` transaksi dengan `High_Login_Risk` (`LoginAttempts >= 3`).
* **Rekomendasi**: Cocok untuk produk *auto-debit*, fitur batas pengeluaran (*budgeting alert*), atau promo *cashback*.

### 1 - Nasabah Mapan Konservatif (`1,653` transaksi / `65.70%`)
* **Profil**: Rata-rata usia `53.71` tahun (dominan `Retired`), saldo tinggi (`$6,955.81`), rata-rata transaksi `$270.63`, dan **`Balance_Ratio` rendah (`0.05`)**. Terdapat `60` transaksi dengan `High_Login_Risk` (`LoginAttempts >= 3`).
* **Rekomendasi**: Nasabah inti dengan dana mengendap besar; tawarkan deposito berjangka, obligasi, atau reksa dana risiko rendah.

### Aturan Risiko Login (`High_Login_Risk = LoginAttempts >= 3`)
* Mencakup **`93` transaksi (`3.70%`)** (`31` masing-masing pada `3`, `4`, dan `5` upaya login) yang tersebar pada kedua segmen (`33` di Cluster 0 dan `60` di Cluster 1).

### Keterbatasan Metodologi
1. **Tanpa label *fraud* asli (*ground truth*)**: proyek ini tidak dapat membuktikan atau mengukur akurasi deteksi *fraud*.
2. **Silhouette Score lemah (`0.2634`)**: batas antar segmen perilaku tidak terpisah tegas karena fitur numerik tumpang tindih.
3. **Indikator berbasis aturan**: `LoginAttempts >= 3` adalah filter aturan eksplisit, bukan prediksi *machine learning*.

## 6.3 Simpan Model K-Means
Simpan `StandardScaler` dan `KMeans` (`k = 2`) ke `.joblib`. Data berlabel lengkap (termasuk kolom anomali) ditulis di langkah **7.3**, setelah deteksi anomali selesai.

In [ ]:
cluster_name_map = {
    0: '0 - Nasabah Muda Aktif',
    1: '1 - Nasabah Mapan Konservatif'
}
df['Segment_Name'] = df['Cluster'].map(cluster_name_map)

scaler_out = 'models/scaler_clustering.joblib' if os.path.isdir('models') else 'scaler_clustering.joblib'
kmeans_out = 'models/kmeans_model.joblib' if os.path.isdir('models') else 'kmeans_model.joblib'

joblib.dump(scaler, scaler_out)
joblib.dump(kmeans_model, kmeans_out)

print("File tersimpan:")
print(f"- {scaler_out}")
print(f"- {kmeans_out}\n")

df.head()

---

# 7. Deteksi Anomali Multivariat (Ensemble 3 Model Unsupervised)
Karena `k = 2` pada K-Means hanya memisahkan profil usia/saldo (Silhouette `0.2634`), tahap ini menambahkan **3 detektor anomali multivariat** pada 8 fitur (termasuk jeda waktu & kecepatan transaksi):
1. **`IsolationForest`**: isolasi berbasis *random tree*.
2. **`LocalOutlierFactor` (`novelty=True`)**: anomali berbasis kepadatan lokal (*density*).
3. **`PCA Reconstruction Error`**: kesalahan rekonstruksi pada `90%` varians.

**Catatan metodologi (dibaca sebelum menafsirkan hasil):**
- **`contamination = 0.05` adalah pilihan, bukan temuan.** Jumlah transaksi yang ditandai ikut angka ini; sensitivitasnya diuji di 7.1.
- **Tanpa label *fraud* asli**, jadi tidak ada precision/recall. Konsensus antar-model hanya menunjukkan *kesepakatan*, bukan kebenaran.
- **`LoginAttempts` ikut menjadi fitur anomali**, sehingga indikator risiko login dan skor anomali saling tumpang tindih (diperiksa di 7.2).
- **`Days_Since_Prev_Tx` perlu dibaca hati-hati** pada data sintetis ini (diperiksa di 7.2).

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

CONTAMINATION = 0.05

assert 'Days_Since_Prev_Tx' in df.columns, "Days_Since_Prev_Tx harus dihitung di langkah 2 (sebelum drop_duplicates)."
df['Amount_Per_Second'] = df['TransactionAmount'] / df['TransactionDuration'].clip(lower=1)

anomaly_features = [
    'TransactionAmount', 'CustomerAge', 'TransactionDuration', 'LoginAttempts',
    'AccountBalance', 'Balance_Ratio', 'Days_Since_Prev_Tx', 'Amount_Per_Second'
]

scaler_anom = StandardScaler()
X_anom = scaler_anom.fit(df[anomaly_features]).transform(df[anomaly_features])

iso = IsolationForest(n_estimators=200, contamination=CONTAMINATION, random_state=42)
iso.fit(X_anom)
iso_pred = iso.predict(X_anom)

lof = LocalOutlierFactor(n_neighbors=20, contamination=CONTAMINATION, novelty=True)
lof.fit(X_anom)
lof_pred = lof.predict(X_anom)

pca_anom = PCA(n_components=0.90, random_state=42)
X_pca_anom = pca_anom.fit_transform(X_anom)
X_rec_anom = pca_anom.inverse_transform(X_pca_anom)
pca_raw = np.mean((X_anom - X_rec_anom) ** 2, axis=1)
pca_threshold = float(np.quantile(pca_raw, 1 - CONTAMINATION))
pca_pred = np.where(pca_raw >= pca_threshold, -1, 1)

df['ISO_Flag'] = iso_pred == -1
df['LOF_Flag'] = lof_pred == -1
df['PCA_Flag'] = pca_pred == -1
df['Anomaly_Votes'] = df['ISO_Flag'].astype(int) + df['LOF_Flag'].astype(int) + df['PCA_Flag'].astype(int)

print(f"PCA: {pca_anom.n_components_} komponen ({pca_anom.explained_variance_ratio_.sum() * 100:.2f}% varians)")
print("\nDistribusi Anomaly_Votes (0..3):")
display(df['Anomaly_Votes'].value_counts().sort_index())
print("\nRata-rata fitur berdasarkan Anomaly_Votes:")
display(df.groupby('Anomaly_Votes')[anomaly_features].mean().round(2))

## 7.0 Skor Anomali Gabungan (0-100)
Skor mentah tiap model diskalakan min-max ke `0-100` lalu dirata-ratakan. Batas min-max (`score_bounds`) disimpan ke bundle agar dashboard memakai skala yang sama persis saat menilai transaksi baru.

In [ ]:
iso_raw = -iso.decision_function(X_anom)
lof_raw = -lof.decision_function(X_anom)

score_bounds = {
    'iso': (float(iso_raw.min()), float(iso_raw.max())),
    'lof': (float(lof_raw.min()), float(lof_raw.max())),
    'pca': (float(pca_raw.min()), float(pca_raw.max())),
}

def minmax_100(values, bounds):
    lo, hi = bounds
    if hi <= lo:
        return np.zeros_like(values, dtype=float)
    return np.clip((values - lo) / (hi - lo), 0.0, 1.0) * 100.0

df['Anomaly_Score'] = (
    minmax_100(iso_raw, score_bounds['iso'])
    + minmax_100(lof_raw, score_bounds['lof'])
    + minmax_100(pca_raw, score_bounds['pca'])
) / 3.0

consensus_mask = df['Anomaly_Votes'] >= 2
print(f"Anomali Konsensus (>= 2 model): {int(consensus_mask.sum())} ({consensus_mask.mean() * 100:.2f}%)")
print(f"Ditandai ketiga model         : {int((df['Anomaly_Votes'] == 3).sum())}")
display(df['Anomaly_Score'].describe().round(2))

## 7.1 Uji Sensitivitas `contamination`
Persentase 5% ditetapkan sendiri. Di sini `contamination` diganti `0.03` / `0.05` / `0.07` untuk melihat seberapa stabil daftar **Anomali Konsensus (>= 2 model)**. Kemiripan dengan daftar dasar (`0.05`) diukur dengan *Jaccard index* (`1.0` = identik).

In [ ]:
def consensus_for(contamination):
    iso_c = IsolationForest(n_estimators=200, contamination=contamination, random_state=42).fit(X_anom)
    lof_c = LocalOutlierFactor(n_neighbors=20, contamination=contamination, novelty=True).fit(X_anom)
    pca_flag_c = pca_raw >= np.quantile(pca_raw, 1 - contamination)
    votes = (
        (iso_c.predict(X_anom) == -1).astype(int)
        + (lof_c.predict(X_anom) == -1).astype(int)
        + pca_flag_c.astype(int)
    )
    return votes >= 2

baseline_mask = consensus_mask.to_numpy()
rows = []
for c in [0.03, 0.05, 0.07]:
    mask = consensus_for(c)
    union = (mask | baseline_mask).sum()
    rows.append({
        'contamination': c,
        'Konsensus (>= 2 model)': int(mask.sum()),
        'Persen (%)': round(mask.mean() * 100, 2),
        'Jaccard vs 0.05': round((mask & baseline_mask).sum() / union, 3) if union else 1.0,
    })

sensitivity_df = pd.DataFrame(rows)
display(sensitivity_df)

## 7.2 Pemeriksaan Keterbatasan Fitur
1. **`Days_Since_Prev_Tx`**: jika `TransactionDate` hanya mencakup rentang sempit, selisih tanggal praktis hanya mencerminkan `PreviousTransactionDate`, bukan perilaku jeda yang bermakna.
2. **Tumpang tindih dengan risiko login**: `LoginAttempts` ada di fitur anomali, jadi cek seberapa besar transaksi `LoginAttempts >= 3` ikut masuk daftar anomali.

In [ ]:
tx_dt = pd.to_datetime(df_raw['TransactionDate'])
prev_dt = pd.to_datetime(df_raw['PreviousTransactionDate'])

print("=== TransactionDate (data mentah) ===")
print(f"Terawal : {tx_dt.min()}")
print(f"Terakhir: {tx_dt.max()}")
print(f"Rentang : {tx_dt.max() - tx_dt.min()}")

gap_raw = (tx_dt - prev_dt).dt.total_seconds() / 86400.0
valid = gap_raw.notna() & prev_dt.notna()
corr_prev = np.corrcoef(gap_raw[valid], prev_dt[valid].astype('int64'))[0, 1]
print(f"\nKorelasi Days_Since_Prev_Tx vs PreviousTransactionDate: {corr_prev:.3f}")
print("(mendekati -1 berarti fitur ini hampir sepenuhnya ditentukan oleh PreviousTransactionDate)")

print("\n=== Risiko Login vs Anomali Konsensus ===")
overlap_tab = pd.crosstab(df['High_Login_Risk'], consensus_mask, margins=True)
overlap_tab.columns = ['Bukan konsensus', 'Konsensus (>= 2 model)', 'Total']
display(overlap_tab)

## 7.3 Simpan Bundle Anomali, Data Final, dan Evaluasi K
Ekspor semua artefak yang dibaca `app.py`:
- `anomaly_bundle.joblib`: scaler, ketiga model, ambang PCA, dan batas skor.
- `data_clustering_clean.csv`: data berlabel lengkap (segmen, flag, skor, votes).
- `k_evaluation.csv`: hasil Elbow/Silhouette `k = 2..8` untuk grafik dashboard.

Sel terakhir mencetak ringkasan angka yang dipakai di README, agar README selalu bisa dicocokkan dengan output notebook.

In [ ]:
anomaly_bundle = {
    'features': anomaly_features,
    'scaler': scaler_anom,
    'iso': iso,
    'lof': lof,
    'pca': pca_anom,
    'pca_threshold': pca_threshold,
    'score_bounds': score_bounds,
    'pca_n_components': int(pca_anom.n_components_),
    'pca_explained_variance': float(pca_anom.explained_variance_ratio_.sum()),
    'contamination': CONTAMINATION,
}

def out_path(folder, name):
    return f"{folder}/{name}" if os.path.isdir(folder) else name

data_out = out_path('data', 'data_clustering_clean.csv')
bundle_out = out_path('models', 'anomaly_bundle.joblib')
keval_out = out_path('models', 'k_evaluation.csv')

df.to_csv(data_out, index=False)
joblib.dump(anomaly_bundle, bundle_out)
eval_df.to_csv(keval_out, index=False)

print("File tersimpan:")
print(f"- {data_out} ({df.shape[0]} baris, {df.shape[1]} kolom)")
print(f"- {bundle_out}")
print(f"- {keval_out}")

# ---- Ringkasan angka untuk README ----
consensus_n = int(consensus_mask.sum())
triple = df[df['Anomaly_Votes'] == 3]
print("\n=== Ringkasan untuk README ===")
print(f"Total transaksi              : {len(df):,}")
print(f"K-Means k=2 Silhouette       : {eval_df.loc[eval_df['k'] == 2, 'Silhouette Score'].iloc[0]:.4f}")
print(f"Segmen 0 / 1                 : {(df['Cluster'] == 0).sum():,} / {(df['Cluster'] == 1).sum():,}")
print(f"PCA                          : {anomaly_bundle['pca_n_components']} komponen, {anomaly_bundle['pca_explained_variance'] * 100:.2f}% varians")
print(f"Anomali Konsensus (>= 2)     : {consensus_n} ({consensus_n / len(df) * 100:.2f}%)")
print(f"Ditandai ketiga model        : {len(triple)} | rata-rata nominal ${triple['TransactionAmount'].mean():,.2f} | Balance_Ratio {triple['Balance_Ratio'].mean():.2f}")
print(f"Risiko login (>= 3)          : {int(df['High_Login_Risk'].sum())} ({df['High_Login_Risk'].mean() * 100:.2f}%)")